In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv


In [2]:
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv")

print(df.shape)
print(df.dtypes)
df.head()

(7043, 21)
customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
# How many customers actually churned?
print(df["Churn"].value_counts())
print()

# Check for missing values
print(df.isnull().sum())
print()

# Look at TotalCharges closely
print(df["TotalCharges"].unique()[:20])

Churn
No     5174
Yes    1869
Name: count, dtype: int64

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

['29.85' '1889.5' '108.15' '1840.75' '151.65' '820.5' '1949.4' '301.9'
 '3046.05' '3487.95' '587.45' '326.8' '5681.1' '5036.3' '2686.05'
 '7895.15' '1022.95' '7382.25' '528.35' '1862.9']


In [4]:
# Find the rows where TotalCharges is just a space
spaces = df[df["TotalCharges"].str.strip() == ""]
print(f"Rows with blank TotalCharges: {len(spaces)}")
print(spaces[["customerID", "tenure", "MonthlyCharges", "TotalCharges"]])

Rows with blank TotalCharges: 11
      customerID  tenure  MonthlyCharges TotalCharges
488   4472-LVYGI       0           52.55             
753   3115-CZMZD       0           20.25             
936   5709-LVOEQ       0           80.85             
1082  4367-NUYAO       0           25.75             
1340  1371-DWPAZ       0           56.05             
3331  7644-OMVMY       0           19.85             
3826  3213-VVOLG       0           25.35             
4380  2520-SGTTA       0           20.00             
5218  2923-ARZLG       0           19.70             
6670  4075-WKNIU       0           73.35             
6754  2775-SEFEE       0           61.90             


In [5]:
# Fix TotalCharges: replace blanks with NaN, then convert to number
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Those 11 rows now have proper NaN — drop them (11 rows out of 7043 is negligible)
df = df.dropna(subset=["TotalCharges"])

# Convert Churn from Yes/No to 1/0 (ML models need numbers)
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

# Drop customerID — it's just a reference number, not useful for prediction
df = df.drop(columns=["customerID"])

# Confirm everything looks right
print(df.shape)
print(df["Churn"].value_counts())
print(df["TotalCharges"].dtype)

(7032, 20)
Churn
0    5163
1    1869
Name: count, dtype: int64
float64


In [6]:
# Churn rate by contract type
print("Churn rate by Contract Type:")
print(df.groupby("Contract")["Churn"].mean().mul(100).round(1).astype(str) + "%")

print()

# Churn rate by whether they have tech support
print("Churn rate by Tech Support:")
print(df.groupby("TechSupport")["Churn"].mean().mul(100).round(1).astype(str) + "%")

print()

# Average monthly charges: churned vs stayed
print("Avg Monthly Charges:")
print(df.groupby("Churn")["MonthlyCharges"].mean().round(2))

print()

# Churn rate by Senior Citizen
print("Churn rate by Senior Citizen (0=No, 1=Yes):")
print(df.groupby("SeniorCitizen")["Churn"].mean().mul(100).round(1).astype(str) + "%")

Churn rate by Contract Type:
Contract
Month-to-month    42.7%
One year          11.3%
Two year           2.8%
Name: Churn, dtype: object

Churn rate by Tech Support:
TechSupport
No                     41.6%
No internet service     7.4%
Yes                    15.2%
Name: Churn, dtype: object

Avg Monthly Charges:
Churn
0    61.31
1    74.44
Name: MonthlyCharges, dtype: float64

Churn rate by Senior Citizen (0=No, 1=Yes):
SeniorCitizen
0    23.7%
1    41.7%
Name: Churn, dtype: object


In [7]:
# What is churn actually costing this business per month?
avg_monthly_charge = df[df["Churn"] == 1]["MonthlyCharges"].mean()
total_churned = df["Churn"].sum()

monthly_revenue_lost = avg_monthly_charge * total_churned
annual_revenue_lost = monthly_revenue_lost * 12

print(f"Customers lost: {total_churned}")
print(f"Avg monthly charge (churned): €{avg_monthly_charge:.2f}")
print(f"Estimated monthly revenue lost: €{monthly_revenue_lost:,.2f}")
print(f"Estimated annual revenue lost: €{annual_revenue_lost:,.2f}")

Customers lost: 1869
Avg monthly charge (churned): €74.44
Estimated monthly revenue lost: €139,130.85
Estimated annual revenue lost: €1,669,570.20


This telecom company is losing €1.67M annually to churn. 42.7% of month-to-month customers leave, compared to just 2.8% on two-year contracts. Churned customers pay €13 more per month than those who stay, suggesting price isn't the issue — service experience is. We built a churn prediction model to identify at-risk customers before they leave, enabling targeted retention campaigns

In [8]:
# Separate target variable from features
X = df.drop(columns=["Churn"])  # Everything except Churn
y = df["Churn"]                 # Just Churn (what we want to predict)

# Convert all remaining text columns to numbers using one-hot encoding
X = pd.get_dummies(X, drop_first=True)

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFirst few column names after encoding:")
print(list(X.columns[:10]))

Features shape: (7032, 30)
Target shape: (7032,)

First few column names after encoding:
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'gender_Male', 'Partner_Yes', 'Dependents_Yes', 'PhoneService_Yes', 'MultipleLines_No phone service', 'MultipleLines_Yes']


In [9]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

# Split: 80% for training, 20% for testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale the numbers so no column dominates just because it's larger
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Train the model
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

# Evaluate it
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1033
           1       0.65      0.57      0.61       374

    accuracy                           0.80      1407
   macro avg       0.75      0.73      0.74      1407
weighted avg       0.80      0.80      0.80      1407

Confusion Matrix:
[[916 117]
 [159 215]]


In a churn model, a false negative — missing a customer who leaves — costs the full lifetime value of that customer. A false positive — wrongly flagging someone — costs only the price of a retention offer, maybe a discount or a call. So we should optimise for recall, accepting some false alarms to catch more real churners

In [10]:
# Tell the model to pay more attention to the minority class (churners)
model_balanced = LogisticRegression(
    max_iter=1000, 
    random_state=42, 
    class_weight="balanced"
)
model_balanced.fit(X_train, y_train)

y_pred_balanced = model_balanced.predict(X_test)
print(classification_report(y_test, y_pred_balanced))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_balanced))

              precision    recall  f1-score   support

           0       0.91      0.70      0.79      1033
           1       0.49      0.80      0.61       374

    accuracy                           0.73      1407
   macro avg       0.70      0.75      0.70      1407
weighted avg       0.79      0.73      0.74      1407

Confusion Matrix:
[[724 309]
 [ 76 298]]


I ran two versions. The default model had 80% accuracy but missed 43% of churners. By setting class_weight to balanced, I dropped accuracy to 73% but caught 80% of actual churners — reducing missed churners from 159 to 76. For a retention campaign, that tradeoff makes business sense because the cost of missing a churner is higher than the cost of a false alarm.

In [11]:
import pandas as pd
import numpy as np

# Get feature importances from the balanced model
feature_names = X.columns
coefficients = model_balanced.coef_[0]

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients
}).sort_values("Coefficient", ascending=False)

print("Top 10 factors INCREASING churn risk:")
print(importance_df.head(10).to_string(index=False))

print("\nTop 10 factors DECREASING churn risk:")
print(importance_df.tail(10).to_string(index=False))

Top 10 factors INCREASING churn risk:
                              Feature  Coefficient
          InternetService_Fiber optic     0.746144
                         TotalCharges     0.623279
                      StreamingTV_Yes     0.259691
                  StreamingMovies_Yes     0.252870
                    MultipleLines_Yes     0.202763
       PaymentMethod_Electronic check     0.193697
                 PaperlessBilling_Yes     0.124250
                        SeniorCitizen     0.075306
                 DeviceProtection_Yes     0.069476
PaymentMethod_Credit card (automatic)     0.029226

Top 10 factors DECREASING churn risk:
                             Feature  Coefficient
     TechSupport_No internet service    -0.091307
DeviceProtection_No internet service    -0.091307
 StreamingMovies_No internet service    -0.091307
                      Dependents_Yes    -0.103848
                     TechSupport_Yes    -0.105289
                  OnlineSecurity_Yes    -0.124312
            

In [12]:
# Add churn probability scores back to the original dataframe
df_output = df.copy()
df_output["Churn_Probability"] = model_balanced.predict_proba(
    scaler.transform(pd.get_dummies(df.drop(columns=["Churn"]), drop_first=True))
)[:, 1]

df_output["Churn_Predicted"] = model_balanced.predict(
    scaler.transform(pd.get_dummies(df.drop(columns=["Churn"]), drop_first=True))
)

df_output["Risk_Segment"] = pd.cut(
    df_output["Churn_Probability"],
    bins=[0, 0.3, 0.6, 1.0],
    labels=["Low Risk", "Medium Risk", "High Risk"]
)

print(df_output["Risk_Segment"].value_counts())
print(df_output[["tenure", "MonthlyCharges", "Churn", "Churn_Probability", "Risk_Segment"]].head(10))

Risk_Segment
Low Risk       2966
High Risk      2390
Medium Risk    1676
Name: count, dtype: int64
   tenure  MonthlyCharges  Churn  Churn_Probability Risk_Segment
0       1           29.85      0           0.810594    High Risk
1      34           56.95      0           0.113912     Low Risk
2       2           53.85      1           0.518938  Medium Risk
3      45           42.30      0           0.078868     Low Risk
4       2           70.70      1           0.851562    High Risk
5       8           99.65      1           0.911778    High Risk
6      22           89.10      0           0.725313    High Risk
7      10           29.75      0           0.565860  Medium Risk
8      28          104.80      1           0.815356    High Risk
9      62           56.15      0           0.029377     Low Risk


In [13]:
# Export enriched dataset for Power BI
df_output.to_csv("telco_churn_with_predictions.csv", index=False)

# Also export feature importance for a dedicated dashboard visual
importance_df.to_csv("feature_importance.csv", index=False)

print("Files exported successfully")
print(f"Main dataset: {df_output.shape[0]} rows, {df_output.shape[1]} columns")
print(f"\nColumns in export:")
print(list(df_output.columns))

Files exported successfully
Main dataset: 7032 rows, 23 columns

Columns in export:
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn', 'Churn_Probability', 'Churn_Predicted', 'Risk_Segment']
